
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fourmodern/molecular_docking_tutorial/blob/main/practical_docking/04_Kinase_Panel.ipynb)

# Kinase Panel — 다중 키나아제 선택성 프로파일링

## 목적

하나의 약물 후보가 **여러 키나아제 중 어떤 것에 선택적으로 결합**하는지 평가합니다.
선택성이 높을수록 부작용이 적은 약물이 될 가능성이 큽니다.

## 이론적 배경

### 키나아제 선택성 (Kinase Selectivity)
인간 게놈에는 ~500개의 키나아제가 있으며, ATP binding site의 구조가 유사합니다.
약물이 타겟 외의 키나아제에도 결합하면 off-target 독성이 발생할 수 있습니다.

### Cross-docking
동일한 리간드를 여러 키나아제 구조에 도킹하여 결합력을 비교합니다.
- 타겟 키나아제에 강하게 결합 + 다른 키나아제에 약하게 결합 → **선택적**
- 모든 키나아제에 비슷하게 결합 → **비선택적 (pan-kinase)**

### 구조 정렬
서로 다른 키나아제라도 ATP binding site 영역은 진화적으로 보존되어 있어,
Cα 원자 기준으로 구조를 정렬하고 포즈를 비교할 수 있습니다.

## 워크플로우
1. 키나아제 패널 정의 (PDB 코드 + 그룹)
2. 전체 구조 준비 & 정렬
3. Docking Box 설정 + 3D 확인
4. Cross-docking (각 키나아제 × 리간드)
5. 선택성 히트맵 & ProLIF 분석
6. 내보내기


## 0. 환경 설정


### 환경 설치

필요한 패키지를 설치합니다.


In [ ]:
#@title 환경 설치 {display-mode: "form"}
import subprocess, sys, os

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + list(pkgs))

pip_install('rdkit', 'meeko', 'gemmi', 'openbabel-wheel')
pip_install('pdbfixer', 'openmm')
pip_install('py3Dmol', 'prolif', 'MDAnalysis')
pip_install('seaborn', 'pandas', 'matplotlib', 'requests')
try: pip_install('pymol-open-source')
except: pass

BIN_DIR = '/opt/bin' if os.path.exists('/opt/bin/smina') else '/content/bin'
os.makedirs(BIN_DIR, exist_ok=True)
smina_path = os.path.join(BIN_DIR, 'smina')
if not os.path.exists(smina_path):
    import stat, urllib.request
    urllib.request.urlretrieve(
        'https://sourceforge.net/projects/smina/files/smina.static/download', smina_path)
    os.chmod(smina_path, os.stat(smina_path).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
os.environ['PATH'] = BIN_DIR + ':' + os.environ['PATH']
print('Done.')


### 라이브러리 로드


In [ ]:
#@title 라이브러리 로드 {display-mode: "form"}
import warnings; warnings.filterwarnings('ignore')
import os, subprocess, urllib.request, time
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import py3Dmol
from rdkit import Chem
from rdkit.Chem import AllChem, Draw, Descriptors, SanitizeFlags
from openbabel import pybel
from meeko import MoleculePreparation, PDBQTWriterLegacy
from pdbfixer import PDBFixer
from openmm.app import PDBFile
import MDAnalysis as mda
from MDAnalysis.analysis import align as mda_align
import prolif as plf

BIN_DIR = '/opt/bin' if os.path.exists('/opt/bin/smina') else '/content/bin'
WORK_DIR = '/content/kinase_panel' if os.path.exists('/content') else os.path.join(os.path.expanduser('~'), 'kinase_panel')
os.makedirs(WORK_DIR, exist_ok=True)
print('All libraries loaded.')


## 1. 키나아제 패널 & 리간드 정의


### 키나아제 패널 정의

비교할 키나아제를 PDB 코드와 함께 정의합니다.
기본 패널은 TK(타이로신 키나아제), CMGC, TKL 그룹에서 대표 5종을 포함합니다.
필요에 따라 추가/제거할 수 있습니다.


In [ ]:
#@title 1-1. 키나아제 패널 정의 {display-mode: "form"}

KINASES = [
    {'name': 'EGFR',  'pdb': '1M17', 'chain': 'A', 'group': 'TK',   'disease': 'NSCLC'},
    {'name': 'ABL1',  'pdb': '1IEP', 'chain': 'A', 'group': 'TK',   'disease': 'CML'},
    {'name': 'JAK2',  'pdb': '3FUP', 'chain': 'A', 'group': 'TK',   'disease': 'MPN'},
    {'name': 'CDK2',  'pdb': '1H1S', 'chain': 'A', 'group': 'CMGC', 'disease': 'Cancer'},
    {'name': 'BRAF',  'pdb': '3OG7', 'chain': 'A', 'group': 'TKL',  'disease': 'Melanoma'},
]

kinase_df = pd.DataFrame(KINASES)
kinase_df.index += 1
print(f'{len(KINASES)} kinases:')
kinase_df


### 도킹할 리간드 정의

모든 키나아제에 동일하게 도킹할 리간드의 SMILES와 이름을 입력합니다.


In [ ]:
#@title 1-2. 도킹할 리간드 정의 {display-mode: "form"}

LIGAND_SMILES = "C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1"  #@param {type:"string"}
LIGAND_NAME = "Erlotinib"  #@param {type:"string"}

mol = Chem.MolFromSmiles(LIGAND_SMILES)
print(f'Ligand: {LIGAND_NAME}')
Draw.MolToImage(mol, size=(300, 200))


## 2. 구조 준비


### 전체 키나아제 구조 준비

각 키나아제에 대해 PDB 다운로드 → 단백질 추출 → 수소 추가 → PDBQT 변환을 수행합니다.
co-crystal 리간드의 좌표도 추출하여 docking box 계산에 사용합니다.


In [ ]:
#@title 2-1. 전체 키나아제 구조 준비 {display-mode: "form"}
structures = {}

for i, k in enumerate(KINASES):
    pdb_id = k['pdb'].lower()
    name = k['name']
    print(f'[{i+1}/{len(KINASES)}] {name} ({pdb_id.upper()})...', end=' ', flush=True)
    
    kdir = os.path.join(WORK_DIR, name)
    os.makedirs(kdir, exist_ok=True)
    
    try:
        # Fetch
        pdb_path = os.path.join(kdir, f'{pdb_id}.pdb')
        if not os.path.exists(pdb_path):
            urllib.request.urlretrieve(f'https://files.rcsb.org/download/{pdb_id}.pdb', pdb_path)
        
        # Extract protein
        u = mda.Universe(pdb_path)
        prot_sel = u.select_atoms(f'protein and chainID {k["chain"]}')
        clean_pdb = os.path.join(kdir, f'{pdb_id}_clean.pdb')
        prot_sel.write(clean_pdb)
        
        # PDBFixer
        prot_H = os.path.join(kdir, f'{pdb_id}_clean_H.pdb')
        fixer = PDBFixer(filename=clean_pdb)
        fixer.findMissingResidues(); fixer.findNonstandardResidues()
        fixer.replaceNonstandardResidues(); fixer.removeHeterogens(True)
        fixer.findMissingAtoms(); fixer.addMissingAtoms(); fixer.addMissingHydrogens(7.4)
        with open(prot_H, 'w') as f:
            PDBFile.writeFile(fixer.topology, fixer.positions, f)
        
        # Receptor PDBQT
        rec_qt = os.path.join(kdir, f'{pdb_id}_rec.pdbqt')
        mol_ob = list(pybel.readfile(format='pdb', filename=prot_H))[0]
        out = pybel.Outputfile(filename=rec_qt+'.tmp', format='pdbqt', overwrite=True)
        out.write(mol_ob); out.close()
        with open(rec_qt+'.tmp') as f: raw = f.readlines()
        with open(rec_qt, 'w') as f:
            for l in raw:
                if not l.startswith(('ROOT','ENDROOT','BRANCH','ENDBRANCH','TORSDOF')): f.write(l)
        os.remove(rec_qt+'.tmp')
        
        # Extract co-crystal ligand for box calculation
        lig_sel = u.select_atoms(f'not protein and not resname HOH WAT SOL GOL PEG EDO PG4 DMS ACT BME CL NA MG ZN CA SO4 and chainID {k["chain"]}')
        if len(lig_sel) < 3:
            lig_sel = u.select_atoms('not protein and not resname HOH WAT SOL GOL PEG EDO PG4 DMS ACT BME CL NA MG ZN CA SO4')
        
        structures[name] = {
            'pdb_id': pdb_id, 'prot_H': prot_H, 'rec_qt': rec_qt,
            'dir': kdir, 'group': k['group'],
            'lig_coords': lig_sel.positions if len(lig_sel) > 0 else None,
        }
        print('OK')
    except Exception as e:
        print(f'FAILED: {e}')

print(f'\n=== {len(structures)}/{len(KINASES)} prepared ===')


## 3. Docking Box 설정


### Docking Box 설정

각 키나아제의 co-crystal 리간드 위치를 기준으로 개별 docking box를 생성합니다.
서로 다른 키나아제이므로 각자의 결합 부위에 맞는 box가 필요합니다.


In [ ]:
#@title 3-1. Box 설정 (키나아제별 auto) {display-mode: "form"}
BOX_METHOD = "auto"  #@param ["auto", "residue", "manual"]
RESIDUE_LIST = "790, 858, 855"  #@param {type:"string"}
MANUAL_CENTER = "12.5, -3.0, 27.0"  #@param {type:"string"}
MANUAL_SIZE = "25.0, 25.0, 25.0"  #@param {type:"string"}
PADDING = 7.0  #@param {type:"number"}

def get_box_from_coords(coords, padding=7.0):
    minC, maxC = coords.min(axis=0), coords.max(axis=0)
    return (
        {'x': float((maxC[0]+minC[0])/2), 'y': float((maxC[1]+minC[1])/2), 'z': float((maxC[2]+minC[2])/2)},
        {'x': float(maxC[0]-minC[0]+2*padding), 'y': float(maxC[1]-minC[1]+2*padding), 'z': float(maxC[2]-minC[2]+2*padding)}
    )

for name, s in structures.items():
    if BOX_METHOD == "auto" and s['lig_coords'] is not None:
        s['center'], s['size'] = get_box_from_coords(s['lig_coords'], PADDING)
    elif BOX_METHOD == "residue":
        res_nums = [int(r.strip()) for r in RESIDUE_LIST.split(',')]
        ref = mda.Universe(s['prot_H'])
        sel = ref.select_atoms(' or '.join([f'resid {r}' for r in res_nums]))
        if len(sel) > 0:
            s['center'], s['size'] = get_box_from_coords(sel.positions, PADDING)
        else:
            s['center'], s['size'] = get_box_from_coords(s['lig_coords'], PADDING)
    elif BOX_METHOD == "manual":
        cx,cy,cz = [float(v) for v in MANUAL_CENTER.split(',')]
        sx,sy,sz = [float(v) for v in MANUAL_SIZE.split(',')]
        s['center'], s['size'] = {'x':cx,'y':cy,'z':cz}, {'x':sx,'y':sy,'z':sz}
    
    print(f'{name}: center=({s["center"]["x"]:.1f}, {s["center"]["y"]:.1f}, {s["center"]["z"]:.1f})')


### Box 3D 시각화

첫 번째 키나아제의 docking box를 3D로 확인합니다.


In [ ]:
#@title 3-2. Box 3D 시각화 (첫 번째 키나아제) {display-mode: "form"}
first_name = list(structures.keys())[0]
s = structures[first_name]

view = py3Dmol.view(width=800, height=600)
with open(s['prot_H']) as f: view.addModel(f.read(), 'pdb')
view.setStyle({'model': 0}, {'cartoon': {'color': 'white', 'opacity': 0.6}})
view.addBox({
    'center': {'x': s['center']['x'], 'y': s['center']['y'], 'z': s['center']['z']},
    'dimensions': {'w': s['size']['x'], 'h': s['size']['y'], 'd': s['size']['z']},
    'color': 'blue', 'opacity': 0.15
})
print(f'{first_name}: Box 시각화')
view.zoomTo()
view.show()


## 4. Cross-docking


In [ ]:
#@title 4-1. 리간드 PDBQT 준비 + 전체 도킹 {display-mode: "form"}
EXHAUSTIVENESS = 16  #@param {type:"integer"}
N_POSES = 5          #@param {type:"integer"}
N_CPUS = 10          #@param {type:"integer"}

smina = os.path.join(BIN_DIR, 'smina')

# Prepare ligand PDBQT
rdmol = Chem.MolFromSmiles(LIGAND_SMILES)
rdmol = Chem.AddHs(rdmol)
AllChem.EmbedMolecule(rdmol, randomSeed=42)
AllChem.MMFFOptimizeMolecule(rdmol)

lig_qt = os.path.join(WORK_DIR, f'{LIGAND_NAME}.pdbqt')
if len(Chem.GetMolFrags(rdmol)) == 1:
    for setup in MoleculePreparation().prepare(rdmol):
        pdbqt_str, is_ok, _ = PDBQTWriterLegacy.write_string(setup)
        if is_ok:
            with open(lig_qt, 'w') as f: f.write(pdbqt_str)
            break

# Dock against each kinase
t0 = time.time()
results = []

for i, (name, s) in enumerate(structures.items()):
    print(f'[{i+1}/{len(structures)}] {name}...', end=' ', flush=True)
    
    sdf_out = os.path.join(s['dir'], f'{LIGAND_NAME}_docked.sdf')
    subprocess.run([
        smina, '-r', s['rec_qt'], '-l', lig_qt, '-o', sdf_out,
        '--center_x', str(s['center']['x']), '--center_y', str(s['center']['y']), '--center_z', str(s['center']['z']),
        '--size_x', str(s['size']['x']), '--size_y', str(s['size']['y']), '--size_z', str(s['size']['z']),
        '--exhaustiveness', str(EXHAUSTIVENESS), '--num_modes', str(N_POSES),
        '--cpu', str(N_CPUS),
    ], stdout=None, stderr=None)
    
    score = None
    if os.path.exists(sdf_out) and os.path.getsize(sdf_out) > 0:
        suppl = list(Chem.SDMolSupplier(sdf_out, removeHs=False))
        if suppl and suppl[0]:
            props = suppl[0].GetPropsAsDict()
            if 'minimizedAffinity' in props:
                try: score = float(props['minimizedAffinity'])
                except: pass
    
    results.append({
        'Kinase': name, 'Group': s['group'],
        'Score': round(score, 2) if score else None,
        'SDF': sdf_out,
    })
    print(f'{score:.2f}' if score else 'FAILED')

elapsed = time.time() - t0
print(f'\n=== Done in {elapsed:.0f}s ===')


## 5. 선택성 분석


### 선택성 프로파일

스코어별로 키나아제를 정렬하여 리간드가 어떤 키나아제에 가장 잘 결합하는지 확인합니다.
- **Selectivity Index**: 최적 스코어 대비 차이. 0에 가까울수록 선택적


In [ ]:
#@title 5-1. 스코어 비교 테이블 & 차트 {display-mode: "form"}
df = pd.DataFrame(results).dropna(subset=['Score']).sort_values('Score')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Score by kinase
group_colors = {'TK': '#E74C3C', 'CMGC': '#3498DB', 'TKL': '#2ECC71', 'AGC': '#F39C12', 'Other': '#9B59B6'}
colors = [group_colors.get(g, 'gray') for g in df['Group']]
axes[0].barh(df['Kinase'], df['Score'], color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_xlabel('Docking Score (kcal/mol)')
axes[0].set_title(f'{LIGAND_NAME} Selectivity Profile')
axes[0].axvline(x=-7.0, color='gray', linestyle='--', alpha=0.5)
for _, row in df.iterrows():
    axes[0].text(row['Score']-0.1, row['Kinase'], f'{row["Score"]:.1f}', va='center', ha='right', fontsize=9, fontweight='bold', color='white')

# Score by group
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=g) for g, c in group_colors.items() if g in df['Group'].values]
axes[0].legend(handles=legend_elements, loc='lower right')

# Selectivity index
best = df['Score'].min()
df['Selectivity'] = df['Score'] - best
axes[1].barh(df['Kinase'], df['Selectivity'], color=colors, edgecolor='black', linewidth=0.5)
axes[1].set_xlabel('ΔScore from best (kcal/mol)')
axes[1].set_title('Selectivity Index (0 = most selective)')

plt.tight_layout()
plt.show()

print(f'\nBest binding: {df.iloc[0]["Kinase"]} ({df.iloc[0]["Score"]:.2f} kcal/mol)')
df[['Kinase', 'Group', 'Score', 'Selectivity']]


### ProLIF 상호작용 비교

각 키나아제에서의 상호작용 유형(수소결합, 소수성 등)을 비교합니다.
선택성의 구조적 근거를 파악할 수 있습니다.


In [ ]:
#@title 5-2. ProLIF 상호작용 비교 {display-mode: "form"}
interaction_data = []

for _, row in df.iterrows():
    name = row['Kinase']
    s = structures[name]
    sdf_out = row['SDF']
    if not os.path.exists(sdf_out): continue
    
    try:
        prot_mol = Chem.MolFromPDBFile(s['prot_H'], removeHs=False, sanitize=False)
        Chem.SanitizeMol(prot_mol, SanitizeFlags.SANITIZE_ALL ^ SanitizeFlags.SANITIZE_PROPERTIES)
        prot_plf = plf.Molecule(prot_mol)
        
        suppl = Chem.SDMolSupplier(sdf_out, sanitize=False, removeHs=False)
        ligs = []
        for mol in suppl:
            if mol is None: continue
            try:
                Chem.SanitizeMol(mol, SanitizeFlags.SANITIZE_ALL ^ SanitizeFlags.SANITIZE_PROPERTIES)
                ligs.append(plf.Molecule(mol))
            except: pass
        
        if ligs:
            fp = plf.Fingerprint()
            fp.run_from_iterable(ligs[:1], prot_plf, n_jobs=1)
            df_ifp = fp.to_dataframe()
            interactions = {}
            for col in df_ifp.columns:
                itype = str(col[2]) if isinstance(col, tuple) and len(col) > 2 else str(col)
                if df_ifp[col].any():
                    interactions[itype] = interactions.get(itype, 0) + 1
            interaction_data.append({
                'Kinase': name, 'Score': row['Score'],
                'HBond': interactions.get('HBDonor', 0) + interactions.get('HBAcceptor', 0),
                'Hydrophobic': interactions.get('Hydrophobic', 0),
                'Total': sum(interactions.values()),
            })
    except Exception as e:
        print(f'{name}: ProLIF failed - {e}')

if interaction_data:
    int_df = pd.DataFrame(interaction_data)
    print('Interaction comparison:')
    int_df


### 포즈 3D 비교

각 키나아제에서의 도킹 포즈를 겹쳐서 3D로 비교합니다.


In [ ]:
#@title 5-3. 포즈 3D 비교 {display-mode: "form"}
view = py3Dmol.view(width=800, height=600)

colors = ['#43A047', '#1E88E5', '#E53935', '#FF8F00', '#8E24AA']
model_idx = 0

for i, (name, s) in enumerate(structures.items()):
    sdf_out = os.path.join(s['dir'], f'{LIGAND_NAME}_docked.sdf')
    if not os.path.exists(sdf_out): continue
    suppl = Chem.SDMolSupplier(sdf_out, removeHs=False, sanitize=False)
    if suppl and suppl[0]:
        block = Chem.MolToMolBlock(suppl[0])
        view.addModel(block, 'mol')
        view.setStyle({'model': model_idx}, {'stick': {'color': colors[i % len(colors)], 'radius': 0.2}})
        model_idx += 1

legend = ', '.join([f'{name}={colors[i%len(colors)]}' for i, name in enumerate(structures.keys())])
print(f'{LIGAND_NAME} in: {legend}')
view.zoomTo()
view.show()


## 6. 내보내기


### 결과 내보내기

CSV + PyMOL 스크립트 + ZIP 파일로 저장합니다.


In [ ]:
#@title 6-1. 결과 내보내기 {display-mode: "form"}
import shutil

csv_path = os.path.join(WORK_DIR, 'kinase_panel_results.csv')
df.to_csv(csv_path, index=False)
print(f'CSV: {csv_path}')

pml_path = os.path.join(WORK_DIR, 'kinase_panel_pymol.pml')
with open(pml_path, 'w') as f:
    f.write('reinitialize\nbg_color white\n\n')
    pml_colors = ['palegreen','lightblue','salmon','lightorange','lightpink']
    for i, (name, s) in enumerate(structures.items()):
        f.write(f'load {s["prot_H"]}, {name}\ncolor {pml_colors[i%len(pml_colors)]}, {name}\nshow cartoon, {name}\n')
        sdf = os.path.join(s['dir'], f'{LIGAND_NAME}_docked.sdf')
        if os.path.exists(sdf):
            f.write(f'load {sdf}, {name}_dock\nset all_states, 0\nframe 1\nshow sticks, {name}_dock\n\n')
print(f'PyMOL: {pml_path}')

zip_path = os.path.join(os.path.dirname(WORK_DIR), 'kinase_panel_results')
shutil.make_archive(zip_path, 'zip', WORK_DIR)
print(f'Archive: {zip_path}.zip')
try:
    from google.colab import files
    files.download(f'{zip_path}.zip')
except ImportError:
    print(f'Results at: {WORK_DIR}')
